# Phase 3 — Traditional ML Baseline

**Objective:**
- Build a strong TF-IDF + LinearSVC baseline.
- Compare with Logistic Regression.
- Evaluate on the validation set.
- Document results before building custom Deep Learning models.

**Important:** Test set is NOT used here.

In [ ]:
import sys, os
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, f1_score
)
import joblib
from utils import load_json

%matplotlib inline
ROOT = Path('..')
print('Ready.')

## 1. Load Train/Val Splits

In [ ]:
df_train = pd.read_csv(ROOT / 'data' / 'train_split.csv')
df_val   = pd.read_csv(ROOT / 'data' / 'val_split.csv')

X_train, y_train = df_train['clean_text'].tolist(), df_train['category'].tolist()
X_val,   y_val   = df_val['clean_text'].tolist(),   df_val['category'].tolist()

print(f'Train: {len(X_train):,}  |  Val: {len(X_val):,}')
print(f'Unique categories: {len(set(y_train))}')

## 2. Load Pre-trained ML Models (from Phase 3 training)

In [ ]:
# Load the saved ML models
lsvc_path = ROOT / 'models' / 'tfidf' / 'pipeline_tf-idf__linearsvc_calibrated.joblib'
lr_path   = ROOT / 'models' / 'tfidf' / 'pipeline_tf-idf__logistic_regression.joblib'

lsvc_pipeline = joblib.load(lsvc_path)
lr_pipeline   = joblib.load(lr_path)

print('Models loaded successfully.')
print(f'LinearSVC pipeline: {lsvc_pipeline}')

## 3. Validation Set Evaluation

In [ ]:
# Evaluate LinearSVC
lsvc_preds = lsvc_pipeline.predict(X_val)
lr_preds   = lr_pipeline.predict(X_val)

print('=== TF-IDF + LinearSVC (Calibrated) ===')
print(f'  Val Accuracy: {accuracy_score(y_val, lsvc_preds):.4f}')
print(f'  Val Macro F1: {f1_score(y_val, lsvc_preds, average="macro", zero_division=0):.4f}')
print(f'  Val Weighted F1: {f1_score(y_val, lsvc_preds, average="weighted", zero_division=0):.4f}')

print('\n=== TF-IDF + Logistic Regression ===')
print(f'  Val Accuracy: {accuracy_score(y_val, lr_preds):.4f}')
print(f'  Val Macro F1: {f1_score(y_val, lr_preds, average="macro", zero_division=0):.4f}')
print(f'  Val Weighted F1: {f1_score(y_val, lr_preds, average="weighted", zero_division=0):.4f}')

## 4. Load Saved Metrics

In [ ]:
ml_metrics = load_json(ROOT / 'artifacts' / 'ml_metrics.json')
print('ML Metrics from training run:')
for model_name, m in ml_metrics['model_metrics'].items():
    print(f"\n  {model_name}")
    for k, v in m.items():
        print(f"    {k}: {v}")

## 5. Confusion Matrix

In [ ]:
# Load pre-saved confusion matrix
cm_path = ROOT / 'results' / 'charts' / 'confusion_matrix_ml_baseline.png'
if cm_path.exists():
    from PIL import Image
    import matplotlib.image as mpimg
    img = mpimg.imread(str(cm_path))
    plt.figure(figsize=(18, 14))
    plt.imshow(img)
    plt.axis('off')
    plt.title('ML Baseline Confusion Matrix (Validation Set)')
    plt.show()
else:
    print('Confusion matrix chart not found. Run train_baseline.py first.')

## 6. ML Model Comparison Chart

In [ ]:
comp_path = ROOT / 'results' / 'charts' / 'ml_model_comparison.png'
if comp_path.exists():
    import matplotlib.image as mpimg
    img = mpimg.imread(str(comp_path))
    plt.figure(figsize=(12, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.show()
else:
    print('Chart not found.')

## Summary

| Model | Val Accuracy | Val Macro F1 | Val Weighted F1 |
|-------|-------------|--------------|----------------|
| TF-IDF + LinearSVC | 0.8741 | 0.8750 | 0.8746 |
| TF-IDF + Logistic Regression | 0.8761 | 0.8755 | 0.8765 |

**Key findings:**
- Both ML models achieve ~87.5% validation accuracy — a strong baseline.
- Logistic Regression slightly outperforms LinearSVC on weighted F1.
- These baselines will be compared against the custom Deep Learning models.

**Next step:** Phase 4-7 — Deep Learning Training and Model Selection